# Chapter 13 — Externalize the Working Set

## Question

**Does information need to remain in the live context merely because the system may need it later?**

Falsifiable version: if large objects move out while stable references stay in, does resident occupancy fall while every payload remains resolvable? If yes, availability and residency are separate variables. Return (what comes back) belongs to Chapter 14; this notebook builds departure only.

## Setup — five working objects, one pressured window

Each object carries stable identity, version, source, representation type, token cost, recoverability, scope, and content. Sizes are deterministic fixtures.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class WorkObject:
    id: str
    version: str
    source: str
    rep_type: str
    tokens: int
    scope: str
    payload: str

OBJECTS = [
    WorkObject('investigation', 'v2', 'agent-trace', 'report', 18000, 'project', 'arch-investigation-payload'),
    WorkObject('benchmark', 'v1', 'tool-run', 'report', 12000, 'project', 'benchmark-payload'),
    WorkObject('trace', 'v4', 'compiler', 'log', 25000, 'project', 'compiler-trace-payload'),
    WorkObject('design', 'v1', 'agent-note', 'note', 15000, 'task', 'design-alternatives-payload'),
    WorkObject('api', 'v7', 'vendor', 'reference', 30000, 'project', 'api-reference-payload'),
]
for o in OBJECTS:
    print(f'{o.id:14s} {o.version:3s} {o.tokens:6d} tokens  scope={o.scope}')
FULL_TOTAL = sum(o.tokens for o in OBJECTS)
print(f'total resident if all full: {FULL_TOTAL}')

## Baseline — everything resident

In [ ]:
live_payloads = {o.id: o.payload for o in OBJECTS}
print(f'resident tokens before: {sum(o.tokens for o in OBJECTS)}')
assert set(live_payloads) == {o.id for o in OBJECTS}

## The artifact store — exact identity only

Resolution happens by exact known identity. No ranking, no embeddings, no search. This is not retrieval.

In [ ]:
class ResolutionFailure(Exception):
    pass

class VersionMismatch(ResolutionFailure):
    pass

@dataclass
class ArtifactStore:
    by_id: dict = field(default_factory=dict)
    location: dict = field(default_factory=dict)

    def write(self, obj, location):
        if obj.id in self.by_id and self.by_id[obj.id].payload != obj.payload:
            raise ResolutionFailure(f'aliasing: {obj.id} already holds a different object')
        self.by_id[obj.id] = obj
        self.location[obj.id] = location

    def exists(self, ref_id):
        return ref_id in self.by_id

    def resolve(self, ref_id, version):
        if ref_id not in self.by_id:
            raise ResolutionFailure(f'orphaned reference: {ref_id} resolves to nothing')
        obj = self.by_id[ref_id]
        if obj.version != version:
            raise VersionMismatch(f'{ref_id}@{version} requested; store holds @{obj.version}')
        return obj.payload

    def move(self, ref_id, new_location):
        self.location[ref_id] = new_location

store = ArtifactStore()
for o in OBJECTS:
    store.write(o, f'/store/{o.id}.json')
print(f'artifacts stored: {len(store.by_id)}')

## Intervention — externalise three objects, keep two resident

The benchmark and trace leave with semantic anchors; the API reference leaves with a bare pointer. The investigation stays COMPACT_RESIDENT; the design note stays FULL_RESIDENT. States are not collapsed: anchor and pointer are different contracts.

In [ ]:
ANCHORS = {
    'benchmark': 'Benchmark: p99 doubles under batch writes. Evidence: artifact://benchmark@v1',
    'trace': 'Trace: 3 failures, assertion mismatch at checkout:42. Evidence: artifact://trace@v4',
}
POINTERS = {'api': 'artifact://api@v7'}
COMPACT = {'investigation': 2500}  # compact resident view, meaning still live

anchor_tokens = sum(len(t) // 4 + 1 for t in ANCHORS.values())
pointer_tokens = sum(len(t) // 4 + 1 for t in POINTERS.values())
resident_after = 15000 + COMPACT['investigation'] + anchor_tokens + pointer_tokens
external_bytes = sum(o.tokens for o in OBJECTS if o.id in ('benchmark', 'trace', 'api'))
print(f'resident before: {FULL_TOTAL}')
print(f'resident after:  {resident_after} (design 15000 + compact 2500 + anchors {anchor_tokens} + pointer {pointer_tokens})')
print(f'external bytes:   {external_bytes}')
print(f'net resident reduction: {FULL_TOTAL - resident_after}')
assert anchor_tokens > pointer_tokens, 'anchors carry meaning; pointers carry identity'
assert resident_after < FULL_TOTAL

live_payloads = {'design': 'design-alternatives-payload'}  # investigation compact view elided as text here
live_references = {**{k: f'{k}@v' for k in ANCHORS}, **POINTERS}

## Departure is not deletion

In [ ]:
originals = {o.id: o.payload for o in OBJECTS}
for aid in ('benchmark', 'trace', 'api'):
    assert aid not in live_payloads, 'payload left the live context'
    assert store.exists(aid), 'artifact preserved outside the live context'
    assert store.resolve(aid, next(o.version for o in OBJECTS if o.id == aid)) == originals[aid]
print('not resident is not gone: every departed payload resolves byte-identical.')

## Integrity failures — explicit, never nearest

An orphaned pointer and a version mismatch. The store surfaces failures; it never silently returns the nearest artifact.

In [ ]:
try:
    store.resolve('incident-99', 'v1')
except ResolutionFailure as e:
    print(f'orphan surfaced: {e}')
    orphan_ok = True

try:
    store.resolve('benchmark', 'v2')
except VersionMismatch as e:
    print(f'version mismatch surfaced: {e}')
    version_ok = True

assert orphan_ok and version_ok
print('Integrity failures are scored as resolution events, never ranking events.')

## Identity versus location — the store moves, the handle holds

In [ ]:
print('location before:', store.location['trace'])
store.move('trace', '/archive/2026/trace-v4.json')
print('location after: ', store.location['trace'])
assert store.resolve('trace', 'v4') == originals['trace']
print('identity artifact:trace@v4 stable across the move.')

## Ledger and cost — departure has a price and a break-even

In [ ]:
ledger = {'artifact_id': 'benchmark@v1', 'source_identity': 'tool-run', 'source_version': 'v1',
          'representation_type': 'report', 'provenance': 'tool-run #41', 'scope': 'project',
          'resident_reference_type': 'anchor', 'resident_reference_tokens': anchor_tokens,
          'external_bytes': 12000, 'recovery_locator': '/store/benchmark.json'}
print('recovery ledger:', ledger)
assert 'recovery_locator' in ledger and 'provenance' in ledger

# Synthetic abstract units: write 2/1k bytes, reference occupancy 1/1k tokens/turn, read 1/1k bytes.
turns = 10
resident_forever = external_bytes * turns
reload_every_turn = external_bytes * 2 // 1000 + turns * ((anchor_tokens + pointer_tokens) + external_bytes)
print(f'10 turns resident-forever: {resident_forever}; externalised-but-reloaded-every-turn: {reload_every_turn}')
assert reload_every_turn > resident_forever
print('Reloaded every turn, departure saves nothing: no universal break-even is claimed.')

## Try it

1. Externalise `design` with a pointer and recompute the reduction — then ask what the next turn loses with no resident meaning.
2. Delete an artifact from the store and re-run departure asserts: the orphan failure must fire, not a silent gap.
3. Bloat an anchor with title, tags, and provenance prose and watch reference tokens eat the savings.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(store.resolve('api', 'v7') == originals['api'])

## What this demonstrates

- Information can remain available to the system without remaining resident in the live context.
- Externalisation differs from pruning because externalisation creates a recovery obligation: every departed payload resolves, and broken references fail explicitly.
- Anchor and pointer are different contracts: meaning plus identity versus identity alone.

## What this does not demonstrate

- That externalisation improves model behaviour or is cheaper than compression.
- That anchors preserve every needed detail, or that a pointer alone suffices.
- That external storage is memory, or that any artifact should be automatically retrieved.
- That version correctness implies world freshness, or that every large object should leave the window.

## Connection to the chapter

Departure is built. Recovery is not:

> Once information has left the live context, the next problem is no longer departure. It is deciding what deserves to come back.

That is Chapter 14.